# Codex agent in Jupyter AI, explanations in nbinlineai

This lesson works with **Jupyter AI 3.2.0** and **nbinlineai 0.1.9 or later** in one JupyterLab environment. Follow [Jupyter AI's setup guide](https://jupyter-ai.readthedocs.io/en/stable/getting-started.html) to install the official `@agentclientprotocol/codex-acp` adapter and Codex CLI, then confirm `codex login status` before starting JupyterLab. Jupyter AI's **Codex** persona works in the separate chat sidebar through that adapter and CLI login. The two **AI Prompt** Markdown cells below belong to nbinlineai; they create paired Markdown answers only when you press their own **Run** buttons with a separately configured nbinlineai API provider. A Codex subscription login does not configure those inline answers. No credentials or generated answers are stored in this notebook.

Run the three Python cells below in order. The first draft deliberately ignores zero values and groups containing only missing values. Its diagnostic cell reports the mismatches without stopping the notebook. Then use the copyable Jupyter AI chat prompts to have Codex fix that one function and rerun the diagnostics. Do **not** use Run All for this lesson: it can also submit unfinished inline AI questions. Jupyter AI's default single-cell tool executes Python code cells, but it does not run nbinlineai's Markdown AI questions.

In [ ]:
# Synthetic daily revenue in dollars: 0.0 is a known zero; None is missing.
sales = [
    {'region': 'North', 'revenue': 10.0},
    {'region': 'North', 'revenue': 0.0},
    {'region': 'North', 'revenue': None},
    {'region': 'North', 'revenue': 20.0},
    {'region': 'South', 'revenue': 5.0},
    {'region': 'South', 'revenue': None},
    {'region': 'South', 'revenue': 15.0},
    {'region': 'South', 'revenue': 0.0},
    {'region': 'East', 'revenue': None},
]
print(f'{len(sales)} synthetic records in three regions')

In [ ]:
def summarize_revenue(rows):
    """Draft summary: count, total, and mean of known revenue by region."""
    grouped = {}
    for row in rows:
        amount = row['revenue']
        if amount:  # BUG: a known zero is false, but should count.
            region = row['region']
            bucket = grouped.setdefault(region, {'count': 0, 'total': 0.0})
            bucket['count'] += 1
            bucket['total'] += amount
    for bucket in grouped.values():
        bucket['mean'] = bucket['total'] / bucket['count']
    return grouped

print(summarize_revenue(sales))

In [ ]:
from math import isclose

expected = {
    'North': {'count': 3, 'total': 30.0, 'mean': 10.0},
    'South': {'count': 3, 'total': 20.0, 'mean': 20.0 / 3.0},
    'East': {'count': 0, 'total': 0.0, 'mean': None},
}


def check_summary():
    """Report every spec mismatch; return True once the draft is fixed."""
    original = [row.copy() for row in sales]
    actual = summarize_revenue(sales)
    problems = []
    if sales != original:
        problems.append('input records changed')
    if set(actual) != set(expected):
        problems.append(f'regions: expected {sorted(expected)}, got {sorted(actual)}')
    for region, want in expected.items():
        got = actual.get(region)
        if got is None:
            continue
        if got.get('count') != want['count']:
            problems.append(f'{region} count: expected {want["count"]}, got {got.get("count")}')
        if not isclose(got.get('total', float('nan')), want['total']):
            problems.append(f'{region} total: expected {want["total"]}, got {got.get("total")}')
        mean = got.get('mean')
        if (mean is None) != (want['mean'] is None) or (mean is not None and not isclose(mean, want['mean'])):
            problems.append(f'{region} mean: expected {want["mean"]}, got {mean}')
    if problems:
        print('Needs work:')
        for problem in problems:
            print(' -', problem)
        return False
    print('All checks pass: zero values count, missing values do not, empty groups remain, and input is unchanged.')
    return True

checks_pass = check_summary()

## Ask the Codex agent in Jupyter AI chat

Open **Jupyter Chat** and select the **Codex** persona. If it asks you to log in, complete `codex login` in a terminal for the same environment and restart JupyterLab. Agent tool permissions belong to Jupyter AI. Copy each prompt into **chat**, not into an nbinlineai AI Prompt cell. The agent should touch only the draft function cell and run only the three named Python cells in order.

**1. Diagnose before editing**

```text
In the open notebook, inspect the Python cells with IDs revenue-data, revenue-summary-draft, and revenue-checks. Explain why the current summary miscounts known zero revenue and omits East. Do not edit any cell or run any notebook command yet.
```

**2. Fix and verify**

```text
In this notebook, edit only the Python cell with ID revenue-summary-draft. Make summarize_revenue count 0.0 as known revenue, ignore None, retain every region including East, and use mean=None when a region has no known values. Preserve the count/total/mean output shape and do not change the input rows. Then run only the Python code cells revenue-data, revenue-summary-draft, and revenue-checks in that order. Report the check cell's actual output. Do not use Run All; do not run, edit, or delete either nbinlineai AI Prompt Markdown cell.
```

If the agent proposes a change without applying it, make the edit yourself and run the three Python cells. The diagnostic should say **All checks pass**. If it does not, inspect the function and retry the checks. Save the notebook after reviewing the change.

## Ask nbinlineai to explain the result

The next two cells are nbinlineai **AI Prompt** Markdown cells. Their own **Run** buttons use nbinlineai's configured API provider; Codex ACP login alone cannot answer them. They are intentionally unrun here. Run the first only if you configured a provider and want an inline answer. The second uses the **Learning** style for a tutoring follow-up. The current live code and earlier notebook text can be used as context according to your Context controls; check those controls before sending if the agent made edits.

Explain in plain language how `summarize_revenue` now treats `0.0` and `None` differently. Use the North and East records as examples, and explain what the check cell verifies. If the draft has not been fixed yet, say that clearly rather than describing an imagined fix.

## Continue as a learner

After the first inline answer appears, run the Learning prompt below. Try answering its question yourself before changing the function. You can edit this Markdown prompt and its paired answer as ordinary notebook notes. Earlier completed inline question/answer pairs can enter later context; Jupyter AI chat history remains in its separate chat file.

Ask me one question that would help me reason about adding a region with two missing values and one known zero. Wait for my attempt before giving a full answer. Which assertions would best show that zero and missing values still have different meanings?